In [6]:
import warnings
warnings.filterwarnings("ignore")

In [1]:
import sys
sys.dont_write_bytecode = True

# from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
PATH = "D://LLM//gemma//gemma3_4b"
# PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# dev

In [2]:
import torch
import numpy as np
import torch.nn.functional as F
from typing import List, Dict, Any
from transformers import DynamicCache

def KVCache_merge(caches: List[DynamicCache]):
    results = DynamicCache()
    # ----- 檢查是否全部都為空 ----- #
    empty_check = [c.key_cache[0] is None for c in caches]
    if np.all(empty_check):
        return results

    # ----- 取出layer ----- #
    seq_len = max([c.get_seq_length(layer_idx=0) for c, is_empty in zip(caches, empty_check) if not is_empty])
    first_non_empty_cache = next(c for c, is_empty in zip(caches, empty_check) if not is_empty)
    n_layers = len(first_non_empty_cache.key_cache)
    n_heads, hid_dim = first_non_empty_cache.key_cache[0].shape[1], first_non_empty_cache.key_cache[0].shape[3]
    
    # ----- 建立cache ----- #
    for i in range(n_layers):
        # ----- 依照不同layer去建立cache ----- #
        keys, values = list(), list()
        for c in caches:
            if c.key_cache[0] is None:
                # ----- 如果是空的，則全部補0 ----- #
                key_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.bfloat16)
                value_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.bfloat16)
                keys += [key_tensor]
                values += [value_tensor]
                continue
            key_tensor = c.key_cache[i]
            value_tensor = c.value_cache[i]

            # ----- 過長的部分做padding(左padding) ----- # 
            curr_seq_len = key_tensor.shape[2]
            if curr_seq_len < seq_len:
                padding_to_add = seq_len - curr_seq_len
                key_tensor = F.pad(key_tensor, (0, 0, padding_to_add, 0), "constant", 0)
                value_tensor = F.pad(value_tensor, (0, 0, padding_to_add, 0), "constant", 0)

            keys += [key_tensor]
            values += [value_tensor]
        
        # ----- merge tensor ----- #
        key_batch = torch.cat(keys, dim=0)
        value_batch = torch.cat(values, dim=0)

        # ----- update cache ----- #
        results.update(key_states=key_batch, value_states=value_batch, layer_idx=i)
    results.seen_tokens = seq_len
    return results

def KVCache_split(cache: DynamicCache, eds: List[int]):
    # ----- 把cache的layer跟數量定義出來 ----- #
    batch_size = cache.key_cache[0].shape[0]
    n_layers = len(cache.key_cache)

    # ----- return的結果 ----- #
    results: List[DynamicCache] = [DynamicCache() for _ in range(batch_size)]

    # ----- by batch操作
    for i in range(batch_size):
        # ----- cache的原始長度，只要用第0層來找即可 ----- #
        """
        因為padding是用0填充，所以如果 hid_dim 和 n_head 都是0，那該位必定padding
        最終找到第一個非零位置
        """
        ed = eds[i]
        sample_key_tensor = cache.key_cache[0][i:i+1] # (1, n_heads, seq_len, hid_dim)
        sum_abs = torch.abs(sample_key_tensor).sum(dim=(1, 3)).squeeze(0)
        non_zero_indices = torch.where(sum_abs > 1e-6)[0] # (seq_len, )

        # ----- seq_len 儲存長度計算 ----- #
        if len(non_zero_indices) == 0: original_seq_len = 0
        else: original_seq_len = non_zero_indices.min().item()
            
        for layer_idx in range(n_layers):
            key_slice = cache.key_cache[layer_idx][i:i+1]
            value_slice = cache.value_cache[layer_idx][i:i+1]

            if ed is not None:
                truncated_key = key_slice[:, :, original_seq_len:ed, :]
                truncated_value = value_slice[:, :, original_seq_len:ed, :]
            else:
                truncated_key = key_slice[:, :, original_seq_len:, :]
                truncated_value = value_slice[:, :, original_seq_len:, :]
            
            results[i].update(
                key_states=truncated_key,
                value_states=truncated_value,
                layer_idx=layer_idx
            )
        
    return results
    

In [6]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration

def prefill_infer(model: Gemma3ForConditionalGeneration,  
          input_ids: List[torch.Tensor], 
          kv_caches: List[DynamicCache]):
    if len(input_ids) == 0:
        return None, []
    try:
        cache = None

        # ----- 整合所有input ----- #
        _input_ids = torch.cat(input_ids, dim=0)
        cache = cache_manager.KVCache_merge(kv_caches)

        # ----- Prefilling過程 ----- #
        with torch.no_grad():
            model(
                input_ids=torch.LongTensor(_input_ids).to(model.device),
                use_cache=True,
                past_key_values=cache,
                )
        print("Prefill Cache Shape:", cache.key_cache[0].shape)
        # ----- 拆解cache ----- #
        eds = []
        for ids in input_ids:
            _list = torch.where(ids==0)[1].tolist()
            if _list:
                eds += [_list[0] - ids.shape[1]]
            else:
                eds += [None]
        caches = cache_manager.KVCache_split(cache,eds)
    finally:
        for item in ("input_ids", ):
            exec(f"del {item}")
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return None, caches


In [13]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration
MAX_NEW_TOKENS_SIZE = 16

def decode_infer(model: Gemma3ForConditionalGeneration,       
                 input_ids: List[torch.Tensor], 
                 kv_caches: List[DynamicCache],
                 uids: List[str],
                 ):
    if len(input_ids) == 0:
        return None, []
    try:
        # ----- 宣告物件 ----- #
        device = model.device
        n_batch = len(kv_caches)
        eos_token_ids = [1, 106] # processor.tokenizer.eos_token_id == 1
        unfinished_sequences = torch.ones(n_batch, dtype=torch.long, device=device)
        generated_ids = {uid: list() for uid in uids}
        merged_cache = cache_manager.KVCache_merge(kv_caches)
        print("Decode Cache Shape:", merged_cache.key_cache[0].shape)

        # ----- 將input做合併 ----- #
        input_ids_tensor = torch.cat(input_ids, dim=0).to(device)

        # ----- Decoding Loop ----- #
        for step in range(MAX_NEW_TOKENS_SIZE):

            # ----- 全部都做完了 ----- #
            if unfinished_sequences.max() == 0:
                break # Stop Decoding
            
            # ----- 計算position_ids ----- #
            cache_len = merged_cache.get_seq_length(layer_idx=0)
            position_ids = torch.tensor([[cache_len-1]], device=device).expand(n_batch, -1)

            # ----- 生成tokens ----- #
            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids_tensor,
                    past_key_values=merged_cache,
                    position_ids=position_ids,
                    use_cache=True)
            logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(logits, dim=-1)
            for i in range(n_batch):
                if unfinished_sequences[i]:
                    generated_ids[uids[i]].append(next_token[i].item())
            input_ids_tensor = next_token.unsqueeze(1)
            is_eos = torch.isin(next_token, torch.tensor(eos_token_ids, device=device))
            unfinished_sequences.mul_(~is_eos) # in-place更新

        # ----- 找EOS位置 ----- #
        eds = list()
        for i in range(n_batch):
            generated_length = len(generated_ids[uids[i]])
            _idx = generated_length - MAX_NEW_TOKENS_SIZE
            eds += [_idx if _idx < 0 else None]

        # ----- Cache更新 ----- #
        new_caches_list = cache_manager.KVCache_split(merged_cache, eds)
 
    finally:
        for item in ("input_ids", "outputs", "logits", "next_token", "token_id"):
            try:
                exec(f"del {item}")
            except:
                pass
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return generated_ids, new_caches_list

# 測試

In [32]:
import uuid
import torch
import importlib
# importlib.reload(scheduler)
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
_scheduler = scheduler.RequestManager()
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    ids = tokenizer.encode(MSG.format(prompt=sentences))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    print("token長度:", len(ids))
    _scheduler.add_request(request)
print()
TEXT = dict()
for _ in range(8):
    d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
    text, DCACHES = decode_infer(llm_model, d_input_ids, d_caches, [r.request_id for r in decode_requests])
    _, PCACHES = prefill_infer(llm_model, p_input_ids, p_caches)
    print(len(PCACHES), len(DCACHES))
    _scheduler.update(PCACHES)
    if text is not None:
        for r in decode_requests:
            _id = r.request_id
            r.input_ids = torch.tensor(text[_id][-1:]).unsqueeze(0)
            r.kv_cache = DCACHES[0]
            _scheduler.add_request(r)
            TEXT[r.request_id] = TEXT.get(r.request_id, "") + tokenizer.decode(text[r.request_id], skip_special_tokens=True)
print("----- 最後輸出 -----\n", TEXT, "\n", "-"*50)

token長度: 29
token長度: 50
token長度: 25
token長度: 24

Prefill Cache Shape: torch.Size([4, 4, 16, 256])
4 0
Prefill Cache Shape: torch.Size([4, 4, 32, 256])
4 0
Prefill Cache Shape: torch.Size([4, 4, 48, 256])
4 0
Decode Cache Shape: torch.Size([3, 4, 29, 256])
Prefill Cache Shape: torch.Size([1, 4, 64, 256])
1 3
Decode Cache Shape: torch.Size([3, 4, 45, 256])
Prefill Cache Shape: torch.Size([1, 4, 65, 256])
1 3
Decode Cache Shape: torch.Size([4, 4, 61, 256])
0 4
Decode Cache Shape: torch.Size([4, 4, 66, 256])
0 4
Decode Cache Shape: torch.Size([4, 4, 82, 256])
0 4
----- 最後輸出 -----
 {'5a8b037f-d9da-4f3a-afaa-6e6f5cad36c0': '半導體廠務主要做的事情，可以概括為以下幾個關鍵步驟，打造出各種電子產品所需的半導體元件：\n\n1. **設計 (Design):**\n   * **晶片設計 (IC Design 說明程與與它的計算細節、原理以及其中蘊含的統計觀念。\n\n**一、Black-Scholes 模型的核心精神**', '8e1ee785-02ad-41ff-baf1-9953fdd2de11': '巨單交易（Mega-Deal Transaction）指的是規模巨大、涉及複雜、更細緻的步驟：\n\n**1. 設計與製程規劃 (設計與製圖)：\n* **晶片設計 (IC Design)：**說明 Black-Scholes 模型，並詳細解釋其計算細節與原理和統計觀念。\n\n**一、Black-Scholes 模型的核心精神', '030ab63d-7da

In [38]:
for k in text:
    print(k, "\n", tokenizer.decode(text[k], skip_special_tokens=True))

679708ad-8c2d-45c2-8ee6-27c6017a3dee 
 觀念。

**一、Black-Scholes 模型的核心精神**


5a8b037f-d9da-4f3a-afaa-6e6f5cad36c0 
 統計觀念。

**一、Black-Scholes 模型的核心精神**
8e1ee785-02ad-41ff-baf1-9953fdd2de11 
 和統計觀念。

**一、Black-Scholes 模型的核心精神
030ab63d-7da7-42d6-9bfc-5dc520720df6 
 統計觀念。

**一、Black-Scholes 模型的核心精神**


In [34]:
text

{'679708ad-8c2d-45c2-8ee6-27c6017a3dee': [239895,
  238630,
  236924,
  108,
  1018,
  237009,
  236951,
  6907,
  236772,
  13790,
  6909,
  228546,
  132542,
  34774,
  1018,
  108],
 '5a8b037f-d9da-4f3a-afaa-6e6f5cad36c0': [125035,
  239895,
  238630,
  236924,
  108,
  1018,
  237009,
  236951,
  6907,
  236772,
  13790,
  6909,
  228546,
  132542,
  34774,
  1018],
 '8e1ee785-02ad-41ff-baf1-9953fdd2de11': [237206,
  125035,
  239895,
  238630,
  236924,
  108,
  1018,
  237009,
  236951,
  6907,
  236772,
  13790,
  6909,
  228546,
  132542,
  34774],
 '030ab63d-7da7-42d6-9bfc-5dc520720df6': [125035,
  239895,
  238630,
  236924,
  108,
  1018,
  237009,
  236951,
  6907,
  236772,
  13790,
  6909,
  228546,
  132542,
  34774,
  1018]}

In [33]:
for t in TEXT:
    print(t, TEXT[t], "\n","-"*50,"\n")

5a8b037f-d9da-4f3a-afaa-6e6f5cad36c0 半導體廠務主要做的事情，可以概括為以下幾個關鍵步驟，打造出各種電子產品所需的半導體元件：

1. **設計 (Design):**
   * **晶片設計 (IC Design 說明程與與它的計算細節、原理以及其中蘊含的統計觀念。

**一、Black-Scholes 模型的核心精神** 
 -------------------------------------------------- 

8e1ee785-02ad-41ff-baf1-9953fdd2de11 巨單交易（Mega-Deal Transaction）指的是規模巨大、涉及複雜、更細緻的步驟：

**1. 設計與製程規劃 (設計與製圖)：
* **晶片設計 (IC Design)：**說明 Black-Scholes 模型，並詳細解釋其計算細節與原理和統計觀念。

**一、Black-Scholes 模型的核心精神 
 -------------------------------------------------- 

030ab63d-7da7-42d6-9bfc-5dc520720df6 機器學習（Machine Learning，簡稱 ML）是一種讓電腦無需明確編程階段，每個階段都有非常複雜的流程和技術：

1. **設計 (Design):**
   * **晶片設計 (IC Design 說明程與與它的計算細節、原理以及其中蘊含的統計觀念。

**一、Black-Scholes 模型的核心精神** 
 -------------------------------------------------- 

679708ad-8c2d-45c2-8ee6-27c6017a3dee 好的，我們來深入探討 Black-Scholes 模型的核心精神、計算細節與原理，並詳細闡述，以及其中蘊含的統計觀念。

**一、Black-Scholes 模型的核心精神**

 
 -------------------------------------------------- 



In [ ]:
d_caches[0].key_cache[0]

tensor([[[[ 0.0491,  0.1279,  0.0679,  ...,  3.8438,  5.8125, -3.6250],
          [-0.4336,  1.8281, -0.0625,  ..., -0.2852, -1.0781,  2.6719],
          [-1.2031,  0.0879, -1.9531,  ...,  4.8438,  4.7188, -1.9375],
          ...,
          [-0.2578,  0.8281, -0.7227,  ..., -0.2852, -1.0781,  2.6719],
          [-0.5469,  0.9414,  0.5039,  ...,  4.3438,  6.1875, -3.9219],
          [-0.1162, -0.1040, -0.2930,  ...,  3.6250,  5.2500, -3.2500]],

         [[ 0.1074,  0.0938, -0.0449,  ...,  0.3066,  0.6875, -0.5117],
          [ 0.2695, -0.4688, -0.3730,  ...,  0.6328,  0.5156,  1.0391],
          [ 1.9766, -0.1680, -0.8984,  ...,  2.0781,  0.1045, -1.1797],
          ...,
          [ 0.3867,  0.9141,  0.0996,  ...,  0.6328,  0.5156,  1.0391],
          [ 0.5664,  4.4688,  0.9297,  ...,  0.4453,  3.5938, -2.5469],
          [-0.2656,  0.2617,  0.0249,  ...,  0.5664,  2.1250, -0.3125]],

         [[-3.2188, -0.3926,  0.0786,  ..., -2.2031,  2.9688,  2.6406],
          [-3.5000,  3.0781, -

: 

In [9]:
_scheduler.PrefillList

OrderedDict()

In [5]:
text

[[240403,
  239525,
  237800,
  238903,
  238758,
  236924,
  108,
  231591,
  238061,
  69592,
  69702,
  103146,
  237283,
  237885,
  239102,
  19966]]

In [35]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt):
    max_seq_len = 64

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                # cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)
                cache_position = torch.arange(past_key_values.get_seq_length(layer_idx=0)-1, 
                                              past_key_values.get_seq_length(layer_idx=0), 
                                              dtype=torch.long, 
                                              device = llm_model.device)
                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                # print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [36]:
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    text1, cache1 = gemma3_resp(sentences)
    print(cache1.key_cache[0].shape)
    print(text1, "\n", "-"*50)

torch.Size([1, 1, 92, 256])
半導體廠務通常在做以下幾個核心任務：

1. **硬件製造：**
   *   製造各種半導體晶片 (如晶片、晶片組件、晶片組件、晶片組件、晶片組裝、晶片組裝、晶片 
 --------------------------------------------------
torch.Size([1, 1, 113, 256])
Black Scholes的核心精神是：**“黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑 
 --------------------------------------------------
torch.Size([1, 1, 88, 256])
巨單交易是指在一個交易中，**所有的交易者（包括交易员）都同意在交易中**，**在交易中，**所有的交易者（包括交易员）都同意**，**交易者（包括交易员）**，**在交易中**，**交易者（包括 
 --------------------------------------------------
torch.Size([1, 1, 87, 256])
機器學習 (Machine Learning) 是一種人工智能 (Artificial Intelligence) 的方法，它利用數據 (Data) 來學習和預測模式 (Patterns) 和決斷 (Predictions) 的能力。

**核心概念：**

*   **數據 (Data):**  機器學習需要收集大量的數據，包含 
 --------------------------------------------------
